# Construindo banco de dados das amostras de vigas

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
import itertools
import numpy as np
import openpyxl
#!pip install parepy-toolbox
from parepy_toolbox import sampling_algorithm_structural_analysis
from obj import momento_limite_armadura_simples, area_aco_flexao_simples, momento_resistente_secao_sem_cor, obj_mestrado_victor

# Planejamento experimental

In [2]:
# Definindo os níveis das variáveis
h_levels = np.linspace(0.30, 0.70, 5)
b_levels = np.linspace(0.14, 0.35, 5)
f_levels = np.linspace(20000, 50000, 5)

# Gerando o planejamento fatorial completo com itertools.product
combinacoes = list(itertools.product(h_levels, b_levels, f_levels))

# Convertendo para DataFrame
df = pd.DataFrame(combinacoes, columns=['h', 'b_w', 'f_ck'])
df

,h,b_w,f_ck
0,0.3,0.14,20000.0
1,0.3,0.14,27500.0
2,0.3,0.14,35000.0
3,0.3,0.14,42500.0
4,0.3,0.14,50000.0
...,...,...,...
120,0.7,0.35,20000.0
121,0.7,0.35,27500.0
122,0.7,0.35,35000.0
123,0.7,0.35,42500.0


# Criando o banco de armaduras

In [3]:
b_w = []
h = []
f_ck = []
m_rdlim = []
pho_s = []
a_s = []
for i in range(len(df)):
    b_w.append(df['b_w'][i])
    h.append(df['h'][i])
    f_ck.append(df['f_ck'][i])
    m_rdlim.append(momento_limite_armadura_simples(df['b_w'][i], df['h'][i], df['f_ck'][i]))
    a_saux, pho_saux = area_aco_flexao_simples(m_rdlim[i], df['b_w'][i], df['h'][i], df['f_ck'][i])
    pho_s.append(pho_saux)
    a_s.append(a_saux)

df['m_rdlim'] = m_rdlim
df['a_s'] = a_s
df['pho_s'] = pho_s
df

,h,b_w,f_ck,m_rdlim,a_s,pho_s
0,0.3,0.14,20000.0,36.584136,0.000380,0.904886
1,0.3,0.14,27500.0,50.303187,0.000523,1.244218
2,0.3,0.14,35000.0,64.022238,0.000665,1.583550
3,0.3,0.14,42500.0,77.741289,0.000808,1.922882
4,0.3,0.14,50000.0,91.460340,0.000950,2.262214
...,...,...,...,...,...,...
120,0.7,0.35,20000.0,497.950740,0.002217,0.904886
121,0.7,0.35,27500.0,684.682268,0.003048,1.244218
122,0.7,0.35,35000.0,871.413795,0.003880,1.583550
123,0.7,0.35,42500.0,1058.145323,0.004711,1.922882


# Subdividindo em grupos por classe de densidade de armadura

In [4]:
b_w = []
h = []
f_ck = []
m_rdlim = []
pho_s = []
a_s = []
divs = 4
for indece, linha in df.iterrows():
    aux = linha['pho_s'] / divs
    pho = 0
    for i in range(divs):
        pho += aux
        pho_s.append(pho)
        b_w.append(linha['b_w'])
        h.append(linha['h'])
        f_ck.append(linha['f_ck'])
        m_rdlim.append(linha['m_rdlim'])
        a_s.append(pho * linha['b_w'] * linha['h'] / 100)

df_aux = {'b_w': b_w, 'h': h, 'f_ck': f_ck, 'm_rdlim': m_rdlim, 'a_s': a_s, 'pho_s': pho_s}
df_aux = pd.DataFrame(df_aux)
df_aux['m_rd'] = df_aux.apply(lambda row: momento_resistente_secao_sem_cor(row['a_s'], row['b_w'], row['h'],row['f_ck']), axis=1)
df_aux

,b_w,h,f_ck,m_rdlim,a_s,pho_s,m_rd
0,0.14,0.3,20000.0,36.584136,0.000095,0.226221,12.162970
1,0.14,0.3,20000.0,36.584136,0.000190,0.452443,22.998372
2,0.14,0.3,20000.0,36.584136,0.000285,0.678664,32.506204
3,0.14,0.3,20000.0,36.584136,0.000380,0.904886,40.686467
4,0.14,0.3,27500.0,50.303187,0.000131,0.311054,16.724084
...,...,...,...,...,...,...,...
495,0.35,0.7,42500.0,1058.145323,0.004711,1.922882,1176.799545
496,0.35,0.7,50000.0,1244.876850,0.001386,0.565554,413.878855
497,0.35,0.7,50000.0,1244.876850,0.002771,1.131107,782.583482
498,0.35,0.7,50000.0,1244.876850,0.004157,1.696661,1106.113881


In [5]:
# df_au = {'b_w': [0.14, 0.15], 'h': [0.3, 0.25], 'f_ck': [20000, 22000], 'm_rd': [36.584, 35], 'a_s': [0.000380, 0.000380], 'pho_s': [0.9048, 0.9048]}
# df = pd.DataFrame(df_au)
# df_aux = df.copy()
# df_aux

# Análise de confiabilidade no tempo

In [6]:
tempos = list(range(0, 101, 20))
beta = []
b_w_aux_list = []
h_aux_list = []
f_ck_aux_list = []
m_rd_aux_list = []
a_s_aux_list = []
mgk_aux_list = []
mqk_aux_list = []
time_aux_list = []
pf_aux_list = []
for i, row in df_aux.iterrows():
    b_w_aux = row['b_w']
    h_aux = row['h']
    f_ck_aux = row['f_ck']
    m_rd_aux = row['m_rd']
    a_s_aux = row['a_s']
    chi_list = [0.15, 0.30, 0.45, 0.60]
    gamma_g = 1.40
    gamma_q = 1.40
    dados_viga = {'h (m)': h_aux, 'b_w (m)': b_w_aux, 'm_rd (kN.m)': m_rd_aux, 'a_s (m2)': a_s_aux, 'gamma_c': 1.00, 'gamma_s': 1.00, 'gamma_f': 1.00}
    none_variable = {'dados_viga': dados_viga, 'time analysis': tempos}

    for id, chi in enumerate(chi_list):
        den_g = gamma_g + gamma_q*chi/(1-chi)
        den_q = gamma_g*(1-chi)/chi + gamma_q
        m_gk = m_rd_aux/den_g
        m_qk = m_rd_aux/den_q

        # Data
        g = {'type': 'normal', 'loc': 1.06*m_gk, 'scale': 0.12*1.06*m_gk, 'stochastic variable': False, 'seed': None}
        q = {'type': 'gumbel max', 'loc': 0.21*m_qk, 'scale': 0.21*0.76*m_qk, 'stochastic variable': True, 'seed': None}
        f_ck = {'type': 'normal', 'loc': 1.22*f_ck_aux, 'scale': 0.15*1.22*f_ck_aux, 'stochastic variable': False, 'seed': None}
        f_yk = {'type': 'normal', 'loc': 1.22*500000, 'scale': 0.04*1.22*500000, 'stochastic variable': False, 'seed': None}
        teta_r = {'type': 'normal', 'loc': 1, 'scale': 0.05, 'stochastic variable': False, 'seed': None}
        teta_s = {'type': 'normal', 'loc': 1, 'scale': 0.05, 'stochastic variable': False, 'seed': None}
        var = [g, q, f_ck, f_yk, teta_r, teta_s]

        # PAREpy setup
        setup = {
                    'number of samples': 40000,
                    'number of dimensions': len(var),
                    'numerical model': {'model sampling': 'mcs-time', 'time steps': len(none_variable['time analysis'])},
                    'variables settings': var,
                    'number of state limit functions or constraints': 1,
                    'none variable': none_variable,
                    'objective function': obj_mestrado_victor,
                    'type process': 'auto',
                    'name simulation': 'victor_teste_inicial',
                }
        # Call algorithm
        results_aux, pf_aux, beta_aux = sampling_algorithm_structural_analysis(setup)
        pf_aux_list_aux = list(pf_aux['I_0'])
        for j in range(len(pf_aux_list_aux)):
            b_w_aux_list.append(b_w_aux)
            h_aux_list.append(h_aux)
            f_ck_aux_list.append(f_ck_aux)
            m_rd_aux_list.append(m_rd_aux)
            mgk_aux_list.append(m_gk)
            mqk_aux_list.append(m_qk)
            a_s_aux_list.append(a_s_aux)
            time_aux_list.append(tempos[j])
            pf_aux_list.append(pf_aux_list_aux[j])
        
        # Supondo que você tenha um DataFrame chamado df_aux
        #results_aux.to_excel("df_aux.xlsx", index=False)

final_df = {
            'b_w': b_w_aux_list,
            'h': h_aux_list,
            'f_ck': f_ck_aux_list,
            'm_rd': m_rd_aux_list,
            'm_gk': mgk_aux_list,
            'm_qk': mqk_aux_list,
            'a_s': a_s_aux_list,
            'time': time_aux_list,
            'pf': pf_aux_list
           }
final_df = pd.DataFrame(final_df)


13:02:23 - Checking inputs completed!
13:02:23 - Started State Limit Function evaluation (g)...
13:02:25 - Finished State Limit Function evaluation (g) in 2.01e+00 seconds!
13:02:25 - Started evaluation beta reliability index and failure probability...
13:02:25 - Finished evaluation beta reliability index and failure probability in 1.88e-02 seconds!
13:02:27 - Voilà!!!!....simulation results are saved in victor_teste_inicial_MCS-TIME_20241202-130225.txt
13:02:27 - Checking inputs completed!
13:02:27 - Started State Limit Function evaluation (g)...
13:02:29 - Finished State Limit Function evaluation (g) in 2.00e+00 seconds!
13:02:29 - Started evaluation beta reliability index and failure probability...
13:02:29 - Finished evaluation beta reliability index and failure probability in 2.57e-02 seconds!
13:02:30 - Voilà!!!!....simulation results are saved in victor_teste_inicial_MCS-TIME_20241202-130229.txt
13:02:30 - Checking inputs completed!
13:02:30 - Started State Limit Function evalua

In [7]:
final_df.to_excel("final_df.xlsx", index=False)

In [8]:
final_df.head(20)

,b_w,h,f_ck,m_rd,m_gk,m_qk,a_s,time,pf
0,0.14,0.3,20000.0,12.16297,7.384661,1.303175,0.000095,0,0.000000
1,0.14,0.3,20000.0,12.16297,7.384661,1.303175,0.000095,20,0.000000
2,0.14,0.3,20000.0,12.16297,7.384661,1.303175,0.000095,40,0.000025
3,0.14,0.3,20000.0,12.16297,7.384661,1.303175,0.000095,60,0.000125
4,0.14,0.3,20000.0,12.16297,7.384661,1.303175,0.000095,80,0.000750
5,0.14,0.3,20000.0,12.16297,7.384661,1.303175,0.000095,100,0.002325
6,0.14,0.3,20000.0,12.16297,6.081485,2.606351,0.000095,0,0.000000
7,0.14,0.3,20000.0,12.16297,6.081485,2.606351,0.000095,20,0.000000
8,0.14,0.3,20000.0,12.16297,6.081485,2.606351,0.000095,40,0.000000
9,0.14,0.3,20000.0,12.16297,6.081485,2.606351,0.000095,60,0.000000
